## [Project] CLIP 기반 박물관 디지털 자산 통합 검색 시스템

### 1. 환경 설정 및 라이브러리 설치
본 프로젝트는 최신 Foundation Model인 CLIP과 LoRA 경량 학습 기술을 활용합니다. 또한 보너스 점수를 위해FAISS(Vector DB)를 연동하여 실시간 검색 성능을 확보합니다.

In [1]:
# 필수 라이브러리 설치
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install peft faiss-cpu

   ---------------------------------------- 0.0/44.8 kB ? eta -:--:--
   --------------------------- ------------ 30.7/44.8 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 44.8/44.8 kB 1.1 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to c:\users\ds\appdata\local\temp\pip-req-build-9kjoyv5m
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369571 sha256=c6e684f925746b7142a552d9b4c87c56c6a430adda76c3192b53f5c60ed59093
  Stored in directory: C:\Users\DS\AppData\Local\Temp\pip-ephem-wheel-cache-pz_6m9x6\wheels\35\3e\df\3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\DS\AppData\Local\Temp\pip-req-build-9kjoyv5m'


   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   --------------------------------------  675.8/680.7 kB 21.5 MB/s eta 0:00:01
   --------------------------------------- 680.7/680.7 kB 21.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.2/18.9 MB 6.3 MB/s eta 0:00:03
   ---------------------------------------- 0.2/18.9 MB 6.3 MB/s eta 0:00:03
    --------------------------------------- 0.3/18.9 MB 2.0 MB/s eta 0:00:10
    --------------------------------------- 0.4/18.9 MB 2.2 MB/s eta 0:00:09
    --------------------------------------- 0.4/18.9 MB 1.8 MB/s eta 0:00:11
   - -------------------------------------- 0.5/18.9 MB 1.9 MB/s eta 0:00:10
   - -------------------------------------- 0.5/18.9 MB 1.8 MB/s eta 0:00:11
   - -------------------------------------- 0.6/18.9 MB 1.7 MB/s eta 0:00:11
   - -------------------------------------- 0.7/18.9 MB 1.6 MB/s eta 0:00:12
   - -

In [2]:
import pandas as pd
import torch
import clip
from peft import LoraConfig, get_peft_model
import faiss
import numpy as np
import time

# 데이터 불러오기
items = pd.read_csv('./items.csv')
items['text_data'] = items['title'].fillna('') + " " + items['description'].fillna('')
print(f"총 {len(items)}개의 박물관 에셋 로드 완료")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# CLIP 모델 로드
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# LoRA 설정 수정
# ViT-B/32 모델의 내부 레이어 명칭에 맞게 target_modules를 수정합니다.
config = LoraConfig(
    r=16,
    lora_alpha=32,
    # 모델 구조에 따라 'visual.proj' 대신 'visual.transformer...' 형태나
    # 혹은 단순히 'out_proj' 등을 타겟팅할 수 있습니다.
    # CLIP 구조에서 가장 범용적인 'out_proj'와 'v_proj'를 타겟팅합니다.
    target_modules=["q_proj", "v_proj", "out_proj"],
    lora_dropout=0.1,
)

# 모델에 LoRA 적용
model = get_peft_model(model, config)
print("LoRA가 적용된 CLIP 모델 구성 완료")

LoRA가 적용된 CLIP 모델 구성 완료


In [ ]:
# 텍스트 임베딩 추출 함수
def get_embeddings(texts):
    tokens = clip.tokenize(texts, truncate=True).to(device)
    with torch.no_grad():
        # peft 모델이므로 model.model을 참조
        return model.model.encode_text(tokens).cpu().numpy().astype('float32')

# # 전체 아이템 인덱싱 (보너스 점수: Vector DB 연동)
# print("Vector DB 인덱싱 시작...")
# batch_size = 64
# all_vecs = []
# for i in range(0, len(items), batch_size):
#     batch = items['text_data'].iloc[i:i+batch_size].tolist()
#     all_vecs.append(get_embeddings(batch))

# all_vecs = np.vstack(all_vecs)
# index = faiss.IndexFlatL2(all_vecs.shape[1])
# index.add(all_vecs)
# print(f"FAISS 인덱스 구축 완료: {index.ntotal}개 벡터 저장")

# 수정: 배치 처리를 통한 고속 인덱싱
print("Vector DB 인덱싱 시작 (GPU 가속 적용)...")
batch_size = 128  # 한 번에 128개씩 처리하여 속도 향상
all_vecs = []

model.model.eval() # 추론 모드로 변경
with torch.no_grad():
    for i in range(0, len(items), batch_size):
        batch_texts = items['text_data'].iloc[i:i+batch_size].tolist()
        # 토큰화 및 GPU 이동
        tokens = clip.tokenize(batch_texts, truncate=True).to(device)
        # 임베딩 추출
        embeddings = model.model.encode_text(tokens)
        all_vecs.append(embeddings.cpu().numpy().astype('float32'))

        if i % 1280 == 0:
            print(f"진행률: {i}/{len(items)} 완료")

all_vecs = np.vstack(all_vecs)
index = faiss.IndexFlatL2(all_vecs.shape[1])
index.add(all_vecs)
print(f"FAISS 인덱스 구축 완료! (총 {index.ntotal}개 에셋)")

Vector DB 인덱싱 시작 (GPU 가속 적용)...
진행률: 0/7562 완료
진행률: 1280/7562 완료
진행률: 2560/7562 완료
진행률: 3840/7562 완료
진행률: 5120/7562 완료
진행률: 6400/7562 완료
✅ FAISS 인덱스 구축 완료! (총 7562개 에셋)


In [ ]:
# 검색 함수 및 실시간성(FPS) 측정
def semantic_search(query, k=5):
    start = time.time()

    # 쿼리 임베딩
    query_vec = get_embeddings([query])

    # Vector DB 검색
    distances, indices = index.search(query_vec, k)

    end = time.time()
    fps = 1 / (end - start)

    return items.iloc[indices[0]], fps

# 테스트 실행
query = "Ancient Egypt"
results, fps = semantic_search(query)
print(f"검색 성능: {fps:.2f} FPS")
display(results[['item_id', 'title', 'modality']])

검색 성능: 5.60 FPS


,item_id,title,modality
6366,v01542,Egypt at the times of V. Golenishchev,video
6380,v01556,Images on Funerary Shrouds. The world of Gods ...,video
6377,v01553,Funerary ritual in Roman Egypt,video
6371,v01547,Radiographic Examinations of the Egyptian Shroud,video
6356,v01508,The Results of Physical and Chemical Studies o...,video


In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install peft faiss-cpu streamlit

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-7fuw394e
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-7fuw394e
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 37.5 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import pandas as pd
import torch
import clip
import faiss
import numpy as np
import time
from peft import LoraConfig, get_peft_model

# [1] 페이지 기본 설정
st.set_page_config(
    page_title="Museum Digital Asset Search",
    page_icon="🏛️",
    layout="wide"
)

# [2] 리소스 로드 (캐싱)
@st.cache_resource
def load_assets():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 모델 로드
    model, preprocess = clip.load("ViT-B/32", device=device)
    
    # LoRA 설정 적용 (초안의 설정 유지)
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "out_proj"],
        lora_dropout=0.1,
    )
    model = get_peft_model(model, config)
    
    # 데이터 로드
    items = pd.read_csv('./items.csv')
    items['text_data'] = items['title'].fillna('') + " " + items['description'].fillna('')
    
    return model, items, device

model, items, device = load_assets()

# [3] FAISS 인덱싱 로직
@st.cache_resource
def get_faiss_index(_items):
    with st.spinner("📦 7,000개 이상의 에셋을 벡터화하고 있습니다..."):
        all_vecs = []
        batch_size = 128
        model.eval()
        
        for i in range(0, len(_items), batch_size):
            batch_texts = _items['text_data'].iloc[i:i+batch_size].tolist()
            tokens = clip.tokenize(batch_texts, truncate=True).to(device)
            with torch.no_grad():
                embeddings = model.model.encode_text(tokens)
                all_vecs.append(embeddings.cpu().numpy().astype('float32'))
        
        vectors = np.vstack(all_vecs)
        index = faiss.IndexFlatL2(vectors.shape[1])
        index.add(vectors)
    return index

index = get_faiss_index(items)

# [4] 검색 함수
def semantic_search(query, k=6):
    start = time.time()
    tokens = clip.tokenize([query], truncate=True).to(device)
    with torch.no_grad():
        query_vec = model.model.encode_text(tokens).cpu().numpy().astype('float32')
    
    distances, indices = index.search(query_vec, k)
    fps = 1 / (time.time() - start)
    return items.iloc[indices[0]], fps

# [5] UI 레이아웃
# 사이드바
with st.sidebar:
    st.image("https://upload.wikimedia.org/wikipedia/commons/thumb/4/4b/Pushkin_Museum_logo.svg/1200px-Pushkin_Museum_logo.svg.png", width=150)
    st.title("System Info")
    st.info(f"Device: {device.upper()}")
    st.write(f"Total Assets: {len(items):,}")
    
    st.divider()
    st.header("Search Settings")
    top_k = st.slider("Result Count", 4, 20, 8)
    
    st.divider()
    fps_display = st.empty()

# 메인 화면
st.title("🏛️ CLIP-LoRA Museum Semantic Search")
st.markdown("---")

query = st.text_input("🔍 검색어를 입력하세요 (예: Ancient Egypt ritual, Golden statue)", placeholder="Search anything...")

if query:
    results, fps = semantic_search(query, k=top_k)
    
    # FPS 실시간 업데이트
    fps_display.metric("Search Speed", f"{fps:.2f} FPS")
    
    st.subheader(f"'{query}'에 대한 {top_k}개의 결과")
    
    # 갤러리 뷰 (2열 구성)
    cols = st.columns(2)
    for idx, (i, row) in enumerate(results.iterrows()):
        with cols[idx % 2]:
            with st.container(border=True):
                # 헤더: 유형 표시 (Video/Image/Object)
                m_color = "red" if row['modality'] == 'video' else "blue"
                st.markdown(f"**:{m_color}[{row['modality'].upper()}]**")
                
                # 제목 및 설명
                st.markdown(f"### {row['title']}")
                with st.expander("상세 설명 보기"):
                    st.write(row['description'] if pd.notna(row['description']) else "설명이 없습니다.")
                
                # 메타데이터
                st.caption(f"ID: {row['item_id']} | Source: Pushkin Museum")
                
                # 링크 버튼
                if pd.notna(row['extra']):
                    st.link_button("박물관 공식 페이지", row['extra'], use_container_width=True)
else:
    # 검색 전 가이드 화면
    st.info("검색어를 입력하면 AI가 의미를 분석하여 가장 적절한 박물관 자산을 찾아줍니다.")
    
    # 추천 키워드 예시
    st.write("💡 추천 검색어:")
    cols = st.columns(4)
    examples = ["Greek Vase", "Renaissance Painting", "Egyptian Mummy", "Artifacts"]
    for i, ex in enumerate(examples):
        if cols[i].button(ex):
            st.rerun()

Overwriting app.py


In [ ]:
# cloudflared 방식 (더 안정적)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup streamlit run app.py &
import time; time.sleep(5)
!./cloudflared tunnel --url http://localhost:8501

nohup: appending output to 'nohup.out'
2026-03-30T03:41:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-30T03:41:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-30T03:42:01Z INF +--------------------------------------------------------------------------------------------+
2026-03-30T03:42:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-30T03:42:01Z INF |  https://rachel